# Mashable Article Popularity Prediction

**Task:** binary classification — will an article get ≥ 1400 shares?  
**Metric:** ROC-AUC. Test set is newer than train (temporal shift).

This notebook covers the full experiment path: hypothesis → code → result → takeaway.  
The final section builds `submission.csv` (the run that ranked 1st among submitted versions).


## 0. Imports and data loading


In [ ]:
import glob
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_predict, StratifiedKFold
from scipy.stats import rankdata

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        hits = glob.glob(f"./**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found")
    return hits[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))

train["publish_date"] = pd.to_datetime(train["publish_date"])
test["publish_date"] = pd.to_datetime(test["publish_date"])
train = train.sort_values("publish_date").reset_index(drop=True)

print(train.shape, test.shape)
print("popular share:", train["target"].mean())
train.head()


## 1. Feature engineering

**Hypothesis:** ratios should lie in [0, 1]; `-1` in keyword fields means missing; log compresses skewed counts; length-normalized ratios and category one-hots add signal.

**What we do:** clip, replace `-1`, `log1p`, `imgs_per_100w` / `links_per_100w` / `self_link_share` / `kw_best_vs_avg`, `month` + `is_weekend`, one-hot `weekday` and `channel`.


In [ ]:
LOG_COLS = [
    "n_tokens_content", "num_hrefs", "num_self_hrefs", "num_imgs", "num_videos",
    "kw_max_min", "kw_avg_min", "kw_min_max", "kw_max_max", "kw_avg_max",
    "kw_min_avg", "kw_max_avg", "kw_avg_avg",
    "self_reference_min_shares", "self_reference_max_shares", "self_reference_avg_shares",
]

def prepare(df):
    d = df.copy()

    for c in ["n_unique_tokens", "n_non_stop_words", "n_non_stop_unique_tokens"]:
        d[c] = d[c].clip(0, 1)

    for c in ["kw_min_min", "kw_avg_min"]:
        d[c] = d[c].where(d[c] >= 0)

    words = d["n_tokens_content"].clip(lower=1)
    d["imgs_per_100w"] = d["num_imgs"] / words * 100
    d["links_per_100w"] = d["num_hrefs"] / words * 100
    d["self_link_share"] = d["num_self_hrefs"] / d["num_hrefs"].clip(lower=1)
    d["kw_best_vs_avg"] = np.log1p(d["kw_max_avg"]) - np.log1p(d["kw_avg_avg"])

    for c in LOG_COLS:
        d[c] = np.log1p(d[c].clip(lower=0))

    d["month"] = d["publish_date"].dt.month
    d["is_weekend"] = d["weekday"].isin(["Saturday", "Sunday"]).astype(int)
    d = pd.get_dummies(d, columns=["weekday", "channel"], dtype=int)

    return d.drop(columns=["publish_date", "id"])

y = train["target"].values
X = prepare(train.drop(columns=["target"]))
X_test = prepare(test).reindex(columns=X.columns, fill_value=0)
print("X shape:", X.shape)


## 2. PCA on keyword and sentiment blocks

**Hypothesis:** correlated blocks can be compressed with little information loss.

**Result:** keywords ≈ 91% variance with 3 components, sentiment ≈ 80% with 4.


In [ ]:
KW_COLS = ["kw_max_min", "kw_avg_min", "kw_min_max", "kw_max_max",
           "kw_avg_max", "kw_min_avg", "kw_max_avg", "kw_avg_avg"]
SENT_COLS = ["global_subjectivity", "global_sentiment_polarity",
             "global_rate_positive_words", "global_rate_negative_words",
             "rate_positive_words", "rate_negative_words",
             "avg_positive_polarity", "min_positive_polarity", "max_positive_polarity",
             "avg_negative_polarity", "min_negative_polarity", "max_negative_polarity"]

for name, cols, n in [("kw", KW_COLS, 3), ("sent", SENT_COLS, 4)]:
    scaler = StandardScaler().fit(X[cols].fillna(0))
    pca = PCA(n_components=n).fit(scaler.transform(X[cols].fillna(0)))
    print(name, "explained variance:", round(pca.explained_variance_ratio_.sum(), 3))
    for df in (X, X_test):
        z = pca.transform(scaler.transform(df[cols].fillna(0)))
        for i in range(n):
            df[f"{name}_pc{i}"] = z[:, i]

print("features after PCA:", X.shape[1])


## 3. Baseline LightGBM (single 80/20 time split)

**Hypothesis:** a conservative model (few leaves, strong regularization) will not overfit noise.

**Result:** ROC-AUC ≈ **0.7518**, ~300 trees.


In [ ]:
split = int(len(X) * 0.8)
X_tr, X_va = X.iloc[:split], X.iloc[split:]
y_tr, y_va = y[:split], y[split:]

params = dict(
    objective="binary", learning_rate=0.03, n_estimators=2000,
    num_leaves=15, min_child_samples=100, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.5, reg_lambda=5, random_state=42, verbose=-1,
)

model = lgb.LGBMClassifier(**params)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)], eval_metric="auc",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
pred_va = model.predict_proba(X_va)[:, 1]
print("Validation ROC-AUC:", roc_auc_score(y_va, pred_va))
print("Best iteration:", model.best_iteration_)


## 4. CatBoost + rank blend

**Hypothesis:** a second model with a different inductive bias + **rank** blending (not raw probabilities) gives a stable gain.

**Result:** CB ≈ 0.7542; best blend at LGBM weight **0.3** → AUC ≈ **0.7545**.

**Takeaway:** strong baseline submission = 0.3 LGBM + 0.7 CB, multiple seeds, full feature set.


In [ ]:
from catboost import CatBoostClassifier

cb = CatBoostClassifier(
    iterations=2000, learning_rate=0.03, depth=6,
    eval_metric="AUC", verbose=0, random_seed=42,
)
cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)

p_cb = cb.predict_proba(X_va)[:, 1]
p_lgb = model.predict_proba(X_va)[:, 1]
print("CatBoost AUC:", roc_auc_score(y_va, p_cb))

for w in [0, 0.3, 0.5, 0.7, 1]:
    blend = w * rankdata(p_lgb) + (1 - w) * rankdata(p_cb)
    print(f"LGBM weight {w}: AUC = {roc_auc_score(y_va, blend):.4f}")


## 5. Adversarial validation

**Hypothesis:** if a model easily separates train from test, there is strong covariate shift; top shift features are candidates for removal or careful handling.

**Result:** train vs test AUC ≈ **0.998**. Top features: `month`, keyword stats.

**Takeaway:** shift exists. But "drop everything that leaks time" only after multi-fold time CV (see section 6).


In [ ]:
adv_X = pd.concat([X, X_test], ignore_index=True)
adv_y = np.r_[np.zeros(len(X)), np.ones(len(X_test))]

adv = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, verbose=-1)
p = cross_val_predict(
    adv, adv_X, adv_y,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method="predict_proba",
)[:, 1]
print("AUC (train vs test):", roc_auc_score(adv_y, p))

adv.fit(adv_X, adv_y)
pd.Series(adv.feature_importances_, index=adv_X.columns).sort_values(ascending=False).head(15)


## 6. Should we drop "temporal" features?

**Hypothesis (single 80/20):** without `month`, `kw_min_min`, `kw_avg_min` validation rises (0.7518 → 0.7549).

**Check:** TimeSeriesSplit, 5 folds — separate noise from real gain.

| Feature set | mean AUC | std |
|-------------|----------|-----|
| all features | ≈ 0.7228 | ≈ 0.023 |
| without month / kw_min_min / kw_avg_min | ≈ 0.7253 | ≈ 0.023 |

**Takeaway:** gain is within noise. **Do not drop these features in the final model.**  
Lesson: a single split overstated the benefit of dropping; the dropped-feature version did worse on the leaderboard.


In [ ]:
def time_block_cv(X, y, model_factory, n_splits=5, feats=None):
    cols = feats if feats is not None else list(X.columns)
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X)):
        m = model_factory()
        m.fit(
            X.iloc[tr_idx][cols], y[tr_idx],
            eval_set=[(X.iloc[va_idx][cols], y[va_idx])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        p = m.predict_proba(X.iloc[va_idx][cols])[:, 1]
        auc = roc_auc_score(y[va_idx], p)
        scores.append(auc)
        print(f"  fold {fold+1}: AUC={auc:.4f}")
    print(f"  mean={np.mean(scores):.4f} ± {np.std(scores):.4f}")
    return scores

lgb_factory = lambda: lgb.LGBMClassifier(**params)

print("LGBM — all features:")
time_block_cv(X, y, lgb_factory, n_splits=5)

DROP = ["month", "kw_min_min", "kw_avg_min"]
FEATS = [c for c in X.columns if c not in DROP]
print("\nLGBM — without month / kw_min_min / kw_avg_min:")
time_block_cv(X, y, lgb_factory, n_splits=5, feats=FEATS)


## 7. Optuna (LightGBM)

**Hypothesis:** hyperparameter search yields a small boost.

**Result:** best on one 80/20 ≈ 0.7548; on TimeSeriesSplit mean ≈ 0.7235 vs 0.7228 for baseline — difference is noise.

**Takeaway:** params are no worse than the original → use them in the final LGBM.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    p = {
        "objective": "binary",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 47),
        "min_child_samples": trial.suggest_int("min_child_samples", 30, 200),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True),
        "n_estimators": 2000,
        "random_state": 42,
        "verbose": -1,
    }
    m = lgb.LGBMClassifier(**p)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)], eval_metric="auc",
        callbacks=[lgb.early_stopping(80, verbose=False)],
    )
    return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("best AUC:", study.best_value)
print("best params:", study.best_params)

best_params = {
    **study.best_params,
    "objective": "binary",
    "n_estimators": 2000,
    "random_state": 42,
    "verbose": -1,
}

print("\nLGBM with Optuna params (time CV):")
time_block_cv(X, y, lambda: lgb.LGBMClassifier(**best_params), n_splits=5)


## 8. Final submission

**Solution that achieved the best public score (1st among 4 uploads):**

- all features (no drop);
- LGBM with `best_params` (Optuna), 5 seeds;
- CatBoost depth=6, 3 seeds;
- rank blend (LGBM weight 0.3 — same as the strong baseline; 0.5 is also fine, val spread ~0.0003);
- fit on full train.

Output file: `submission.csv`.


In [ ]:
W = 0.3  # LGBM weight in rank blend

p_lgb_test = np.zeros(len(X_test))
for seed in range(5):
    m = lgb.LGBMClassifier(**{**best_params, "random_state": seed})
    m.fit(X, y)
    p_lgb_test += m.predict_proba(X_test)[:, 1] / 5

p_cb_test = np.zeros(len(X_test))
for seed in range(3):
    c = CatBoostClassifier(
        iterations=800, learning_rate=0.03, depth=6,
        verbose=0, random_seed=seed,
    )
    c.fit(X, y)
    p_cb_test += c.predict_proba(X_test)[:, 1] / 3

pred_test = (W * rankdata(p_lgb_test) + (1 - W) * rankdata(p_cb_test)) / len(X_test)

sample = pd.read_csv(find("sample_submission.csv"))
sub = pd.DataFrame({sample.columns[0]: test["id"], sample.columns[1]: pred_test})
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.head()


## Experiment summary

| Step | Decision | Effect |
|------|----------|--------|
| Prep + PCA | clip, log, ratios, one-hot, PCA | working baseline |
| LGBM 80/20 | conservative params | ~0.752 |
| + CatBoost rank blend | 0.3/0.7 | ~0.754–0.755 |
| Adversarial + feature drop | FEATS on one split | local +0.003, worse on LB |
| TimeSeriesSplit | 5 folds | drop **not** confirmed |
| Optuna | tune LGBM | ≈0 on time-CV, params OK |
| Final | all features, seed avg, rank blend | **best submission, 1st place** |

Main methodological takeaway: under temporal shift, feature decisions must rely on **several** time folds, not a single 80/20 split.
